# EDA e hipótesis preregistradas

Especificación 003 v1.1. Este notebook presenta descriptivos de entrenamiento y validación, los resultados ya calculados y las seis figuras. Las funciones estadísticas están en `src/churn/`; aquí no se definen funciones ni se ajustan modelos.

Preparación desde la raíz: `uv sync --locked --all-groups`, `uv run churn-split`, `uv run churn-hypotheses` y `uv run --group eda python -m churn.plots`. Selecciona el intérprete de `.venv` para ejecutar las celdas.

El conjunto de prueba permanece reservado. El CSV no se distribuye en Git y todas las salidas del notebook se eliminan antes de versionarlo.

In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

from churn.config import PROJECT_ROOT
from churn.split import load_exploration

df = load_exploration()
display(df.groupby("partition", observed=True).size().to_frame("filas"))

## Tipos, cardinalidades y distribuciones

Los identificadores se muestran únicamente para auditar cardinalidades; no son entradas de un modelo. Los estadísticos siguientes pertenecen a las 8.000 filas de exploración. El reporte global de calidad de la semana 1 es una limitación documentada y no se reutiliza para los contrastes.

In [ ]:
display(pd.DataFrame({"tipo": df.dtypes.astype(str), "cardinalidad": df.nunique()}))
display(df.select_dtypes(include="number").describe().T)
display(df.select_dtypes(include=["string", "object"]).describe().T)

## Resultados de H1–H6

La tabla se carga de `reports/hypotheses.json`; no se recalculan pruebas en el notebook. Holm corresponde solo a H1–H5. H6 se evalúa por equivalencia del IC de AUC dentro de [0,45; 0,55]. Los IC de H2 son los de las tasas por nivel, disponibles en los detalles del JSON, no un IC de V de Cramér.

In [ ]:
results = json.loads((PROJECT_ROOT / "reports" / "hypotheses.json").read_text(encoding="utf-8"))
display(
    pd.DataFrame(results)[
        ["id", "prueba_usada", "efecto", "ic_low", "ic_high", "p_holm", "veredicto"]
    ]
)

## Figuras

Las seis imágenes se generan con `churn.plots`, que dibuja los efectos e intervalos del reporte. En la figura de productos, 3–4 es un nivel agrupado conforme a H2. La distribución de saldo separa explícitamente la masa puntual en cero.

In [ ]:
for filename in (
    "churn_products.png",
    "churn_age.png",
    "germany_or.png",
    "balance_distribution.png",
    "salary_distribution.png",
    "churn_activity.png",
):
    display(Image(filename=str(PROJECT_ROOT / "reports" / "figures" / filename)))

## Lectura y límites

Los contrastes describen asociaciones en este dataset y no efectos de intervenciones. Las conclusiones e implicaciones para la semana 3 se documentan en `reports/eda_hypotheses.md`. La evaluación final sobre prueba, la selección de variables y los modelos predictivos corresponden a fases posteriores.